In [1]:
import glob
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil as sh

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
RUTA_BASE = "/content/drive/MyDrive/Tesis/vasijas"

CLASE = 4

RUTA_CLASIFICADOS = RUTA_BASE + "/Clasificados"
RUTA_MITADES = RUTA_BASE + "/Mitades"

perfiles = glob.glob(
    f'{RUTA_CLASIFICADOS}/Clase{CLASE:02d}/*.png'
)

vertical = CLASE in range(1, 10)

In [4]:
def mostrar_imagen(imagen):
    plt.imshow(imagen, cmap=plt.cm.gray)
    plt.axis("on")
    plt.figure()


def cortar_imagen(img, vertical):
    alto, ancho = img.shape
    mitad_vertical = alto // 2
    mitad_horizontal = ancho // 2

    fondo_base = np.full(img.shape, 255, dtype=np.uint8)
    fondo_tope = np.full(img.shape, 255, dtype=np.uint8)

    if vertical:

        # Tope
        fondo_tope[:mitad_vertical, :] = img[:mitad_vertical, :]
        # Base
        fondo_base[mitad_vertical:, :] = img[mitad_vertical:, :]

    else:

        # Tope
        fondo_tope[:, :mitad_horizontal] = img[:, :mitad_horizontal]
        # Base
        fondo_base[:, mitad_horizontal:] = img[:, mitad_horizontal:]

    return (fondo_tope, fondo_base)

def guardar_mitad_imagen(
    ruta_imagen_original: str,
    imagen_mitad,
    sufijo: str,
    clase: int,
    ruta_mitades: str
):
    """
    Guarda una mitad de imagen en la carpeta correspondiente a su clase.

    Ejemplo de salida:
    RUTA_MITADES/Clase11/AL_03A_tope.png
    """

    # 1. Nombre del archivo sin extensión
    nombre_archivo = os.path.splitext(
        os.path.basename(ruta_imagen_original)
    )[0]

    # 2. Carpeta de la clase (Clase01, Clase11, etc.)
    carpeta_clase = f"Clase{clase:02d}"
    ruta_clase = os.path.join(ruta_mitades, carpeta_clase)

    # 3. Crear carpeta si no existe
    os.makedirs(ruta_clase, exist_ok=True)

    # 4. Ruta final del archivo
    ruta_salida = os.path.join(
        ruta_clase,
        f"{nombre_archivo}{sufijo}.png"
    )

    # 5. Guardar imagen
    cv2.imwrite(ruta_salida, imagen_mitad)

    return ruta_salida

def leer_imagen(ruta_imagen: str):
    img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    return img

In [5]:
# Crear carpeta base de mitades si no existe
os.makedirs(RUTA_MITADES, exist_ok=True)

for ruta_imagen in perfiles:

    img = leer_imagen (ruta_imagen)

    tope, base = cortar_imagen(img, vertical)

    # mostrar_imagen(tope)
    # mostrar_imagen(base)

    guardar_mitad_imagen(
        ruta_imagen_original = ruta_imagen,
        imagen_mitad = tope,
        sufijo = "_tope",
        clase = CLASE,
        ruta_mitades = RUTA_MITADES
    )

    guardar_mitad_imagen(
        ruta_imagen_original = ruta_imagen,
        imagen_mitad = base,
        sufijo = "_base",
        clase = CLASE,
        ruta_mitades = RUTA_MITADES
    )